# Vision-Language Grounding System Demo

This notebook demonstrates the capabilities of the Vision-Language Grounding System, including:
1. Model architecture overview
2. Training visualization
3. Zero-shot classification
4. Image-text retrieval
5. Visual grounding
6. Attention visualization

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

# Import project modules
from config import get_config
from models import create_vlg_model, SimpleTokenizer
from data import get_transforms
from utils.visualization import (
    visualize_attention_map,
    visualize_similarity_matrix,
    visualize_grounding
)

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Model Architecture Overview

In [ ]:
# Load configuration
config = get_config()

# Create model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = create_vlg_model(config)
model = model.to(device)
model.eval()

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
vision_params = count_parameters(model.vision_encoder)
text_params = count_parameters(model.text_encoder)
fusion_params = count_parameters(model.fusion)

print(f"\nModel Parameter Count:")
print(f"  Total: {total_params / 1e6:.2f}M")
print(f"  Vision Encoder: {vision_params / 1e6:.2f}M")
print(f"  Text Encoder: {text_params / 1e6:.2f}M")
print(f"  Fusion Module: {fusion_params / 1e6:.2f}M")

# Print model structure
print("\nModel Architecture:")
print(model)

## 2. Test Forward Pass

In [ ]:
# Create dummy inputs
batch_size = 4
images = torch.randn(batch_size, 3, 224, 224).to(device)
input_ids = torch.randint(0, 49408, (batch_size, 77)).to(device)

# Forward pass
with torch.no_grad():
    outputs = model(images, input_ids, compute_grounding=True, return_attention=True)

print("Forward pass successful!")
print("\nOutput shapes:")
for key, value in outputs.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")

## 3. Zero-Shot Classification Demo

In [ ]:
# Create sample image (replace with real image in practice)
sample_image = torch.randn(1, 3, 224, 224).to(device)

# Define class descriptions
class_names = ["cat", "dog", "bird", "car", "person", "bicycle", "tree", "building"]
template = "a photo of a {}"
class_descriptions = [template.format(cls) for cls in class_names]

# Tokenize class descriptions
tokenizer = SimpleTokenizer()
class_input_ids = tokenizer.encode(class_descriptions, device=device)

# Zero-shot classification
with torch.no_grad():
    # Encode image and texts
    image_features, _ = model.encode_image(sample_image)
    text_features, _ = model.encode_text(class_input_ids)
    
    # Compute similarity
    similarity = model.fusion.get_image_text_similarity(image_features, text_features)
    
    # Get predictions
    probs = F.softmax(similarity[0], dim=0).cpu().numpy()
    predicted_idx = probs.argmax()

# Visualize results
plt.figure(figsize=(12, 6))

# Plot probabilities
plt.subplot(1, 2, 1)
plt.barh(class_names, probs)
plt.xlabel('Probability')
plt.title('Zero-Shot Classification Scores')
plt.axvline(x=probs[predicted_idx], color='r', linestyle='--', label='Predicted')
plt.legend()

# Plot similarity matrix
plt.subplot(1, 2, 2)
plt.imshow(similarity.cpu().numpy(), cmap='coolwarm', aspect='auto')
plt.colorbar(label='Similarity Score')
plt.xlabel('Text Classes')
plt.ylabel('Image')
plt.title('Image-Text Similarity')
plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')

plt.tight_layout()
plt.show()

print(f"\nPredicted class: {class_names[predicted_idx]}")
print(f"Confidence: {probs[predicted_idx]:.4f}")

## 4. Image-Text Retrieval Demo

In [ ]:
# Create sample images and texts
num_images = 5
num_texts = 5

images = torch.randn(num_images, 3, 224, 224).to(device)
texts = [
    "a cat sitting on a mat",
    "a dog playing in a park",
    "a bird flying in the sky",
    "a car on the street",
    "a person walking"
]

# Encode
text_input_ids = tokenizer.encode(texts, device=device)

with torch.no_grad():
    image_features, _ = model.encode_image(images)
    text_features, _ = model.encode_text(text_input_ids)
    
    similarity_matrix = model.fusion.get_image_text_similarity(
        image_features, text_features
    )

# Visualize similarity matrix
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix.cpu().numpy(), cmap='coolwarm', aspect='auto')
plt.colorbar(label='Similarity Score')
plt.xlabel('Text Queries')
plt.ylabel('Images')
plt.title('Image-Text Similarity Matrix')
plt.xticks(range(len(texts)), [f"T{i}" for i in range(len(texts))])
plt.yticks(range(len(images)), [f"I{i}" for i in range(len(images))])

# Add text labels
for i in range(len(texts)):
    plt.text(i, -0.7, texts[i][:20] + "...", ha='center', va='top', 
             rotation=45, fontsize=8)

plt.tight_layout()
plt.show()

# Show top-k retrieval
k = 3
print(f"\nTop-{k} Image-to-Text Retrieval:")
for i in range(min(3, num_images)):
    top_k_indices = similarity_matrix[i].topk(k).indices.cpu().numpy()
    print(f"  Image {i}:")
    for rank, idx in enumerate(top_k_indices, 1):
        score = similarity_matrix[i, idx].item()
        print(f"    {rank}. [{score:.4f}] {texts[idx]}")

## 5. Visual Grounding Demo

In [ ]:
# Create sample image and text query
sample_image = torch.randn(1, 3, 224, 224).to(device)
text_query = "the cat"

# Encode
query_input_ids = tokenizer.encode([text_query], device=device)

with torch.no_grad():
    # Get patch features and token features
    _, patch_features = model.encode_image(sample_image, return_all_tokens=True)
    _, token_features = model.encode_text(query_input_ids, return_all_tokens=True)
    
    # Remove CLS token from patches
    patch_features_only = patch_features[:, 1:, :]
    
    # Compute grounding through cross-attention
    grounded_features, grounding_attn = model.fusion.grounding_module(
        token_features,
        patch_features_only,
        return_attention=True
    )
    
    # Average attention over heads and tokens to get patch scores
    grounding_scores = grounding_attn.mean(dim=(1, 2))[0]

# Visualize grounding
num_patches = int(np.sqrt(len(grounding_scores)))
score_map = grounding_scores.cpu().numpy().reshape(num_patches, num_patches)

plt.figure(figsize=(12, 4))

# Original "image" (random noise in this case)
plt.subplot(1, 3, 1)
img_display = sample_image[0].permute(1, 2, 0).cpu().numpy()
img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())
plt.imshow(img_display)
plt.title("Input Image")
plt.axis('off')

# Grounding heatmap
plt.subplot(1, 3, 2)
plt.imshow(score_map, cmap='hot', interpolation='bilinear')
plt.colorbar(label='Grounding Score')
plt.title("Grounding Heatmap")
plt.axis('off')

# Cross-attention visualization
plt.subplot(1, 3, 3)
attn_avg = grounding_attn[0].mean(0).cpu().numpy()  # Average over heads
plt.imshow(attn_avg, cmap='viridis', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Image Patches')
plt.ylabel('Text Tokens')
plt.title(f'Cross-Attention\nQuery: "{text_query}"')

plt.tight_layout()
plt.show()

print(f"\nGrounding scores for '{text_query}':")
print(f"  Max score: {grounding_scores.max().item():.4f}")
print(f"  Min score: {grounding_scores.min().item():.4f}")
print(f"  Mean score: {grounding_scores.mean().item():.4f}")

## 6. Attention Visualization

In [ ]:
# Get attention from vision encoder
sample_image = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    _, _, vision_attn = model.vision_encoder(
        sample_image,
        return_all_tokens=True,
        return_attention=True
    )

# Visualize different attention heads
num_heads_to_show = 4
fig, axes = plt.subplots(2, num_heads_to_show, figsize=(16, 8))

for i in range(num_heads_to_show):
    # Get attention for this head (CLS token attending to all tokens)
    head_attn = vision_attn[0, i, 0, 1:].cpu().numpy()  # Remove CLS-to-CLS
    
    # Reshape to grid
    num_patches = int(np.sqrt(len(head_attn)))
    attn_map = head_attn.reshape(num_patches, num_patches)
    
    # Show attention map
    axes[0, i].imshow(attn_map, cmap='hot', interpolation='bilinear')
    axes[0, i].set_title(f'Head {i} - CLS Attention')
    axes[0, i].axis('off')
    
    # Show attention pattern (all-to-all for center patch)
    center_patch = len(head_attn) // 2
    patch_attn = vision_attn[0, i, center_patch + 1, 1:].cpu().numpy()
    patch_attn_map = patch_attn.reshape(num_patches, num_patches)
    
    axes[1, i].imshow(patch_attn_map, cmap='viridis', interpolation='bilinear')
    axes[1, i].set_title(f'Head {i} - Center Patch')
    axes[1, i].axis('off')

plt.suptitle('Vision Transformer Attention Patterns', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Average attention across all heads
avg_attn = vision_attn[0].mean(0)[0, 1:].cpu().numpy()
avg_attn_map = avg_attn.reshape(num_patches, num_patches)

plt.figure(figsize=(8, 6))
plt.imshow(avg_attn_map, cmap='hot', interpolation='bilinear')
plt.colorbar(label='Attention Weight')
plt.title('Average Attention Across All Heads\n(CLS token attending to patches)', 
          fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 7. Training Simulation (Small Example)

In [ ]:
# Create a small training example
print("Simulating a mini training step...")

# Create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Training mode
model.train()

# Create a batch
batch_images = torch.randn(8, 3, 224, 224).to(device)
batch_texts = tokenizer.encode(
    ["sample text"] * 8,
    device=device
)

# Forward pass
losses = model.compute_loss(batch_images, batch_texts)

print("\nLoss breakdown:")
print(f"  Total loss: {losses['total_loss'].item():.4f}")
print(f"  Contrastive loss: {losses['contrastive_loss'].item():.4f}")
print(f"  Image-to-text loss: {losses['loss_i2t'].item():.4f}")
print(f"  Text-to-image loss: {losses['loss_t2i'].item():.4f}")

# Backward pass
optimizer.zero_grad()
losses['total_loss'].backward()
optimizer.step()

print("\nTraining step completed successfully!")

# Back to eval mode
model.eval()

## 8. Model Summary and Next Steps

In [ ]:
print("=" * 60)
print("VISION-LANGUAGE GROUNDING SYSTEM SUMMARY")
print("=" * 60)

print("\n📊 Model Architecture:")
print(f"  ✓ Vision Encoder: ViT-Base/16 ({vision_params / 1e6:.2f}M params)")
print(f"  ✓ Text Encoder: Transformer ({text_params / 1e6:.2f}M params)")
print(f"  ✓ Fusion Module: Contrastive + Cross-Attention ({fusion_params / 1e6:.2f}M params)")
print(f"  ✓ Total Parameters: {total_params / 1e6:.2f}M")

print("\n🎯 Capabilities Demonstrated:")
print("  ✓ Zero-shot image classification")
print("  ✓ Image-to-text retrieval")
print("  ✓ Text-to-image retrieval")
print("  ✓ Visual grounding with cross-attention")
print("  ✓ Attention visualization")
print("  ✓ Training loop (simulated)")

print("\n🚀 Next Steps:")
print("  1. Train on real dataset (COCO Captions or Flickr30k)")
print("  2. Fine-tune on downstream tasks")
print("  3. Evaluate on standard benchmarks")
print("  4. Experiment with different architectures")
print("  5. Add more sophisticated visual grounding")

print("\n📚 Resources:")
print("  • Training: python train.py")
print("  • Inference: python inference.py --checkpoint <path>")
print("  • Evaluation: python evaluate.py --checkpoint <path>")
print("  • Tests: pytest tests/ -v")

print("\n" + "=" * 60)
print("Demo completed successfully! 🎉")
print("=" * 60)